## Tools
Models can request to call tools that perform tasks such as fetching data from a database,searching the web, or running code. Tools are pairings of:
*  1.A schema,including the name of the tool,a description and argumrnt definitions(often a JSON schema)
*  2.A function or coroutine to execute

In [18]:
import os 
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

os.environ["Groq_API_Key"]= os.getenv("Groq_API_Key")

model = init_chat_model(
    model="openai/gpt-oss-20b",
    model_provider="groq",
)

response = model.invoke(
    [{"role": "user", "content": "Why do parrots talk?"}]
)

response.content



'Parrots aren’t “talking” in the human sense—they’re excellent mimics.  Their ability to repeat words, phrases, and even the intonation of a voice is the result of a combination of biology, evolution, and environment.  Here’s why they do it:\n\n| What | Why it happens | What it means for us |\n|------|----------------|-----------------------|\n| **Vocal learning** | Parrots have a highly developed *vocal learning* system, similar to humans, that lets them map sounds they hear onto their own vocal apparatus. | They can reproduce a wide range of sounds, from bird calls to human speech. |\n| **Social bonding** | In the wild, parrots live in large flocks that rely on complex vocal communication to maintain group cohesion, locate food, and warn of predators. | Mimicking a human voice can be a way of “talking” to you as if you were another flock member. |\n| **Environmental enrichment** | A quiet, repetitive environment (like a cage) can lead to boredom. Parrots use vocal mimicry to fill the

In [22]:
from langchain.tools import tool
@tool
def get_weather(location: str) -> str:
    """
    Get the current weather for a given location.
    """
    # For demonstration purposes, we'll return a static response.
    # In a real implementation, you would call a weather API here.
    return f"The current weather in {location} is sunny with a temperature of 25°C."

model_with_tools = model.bind_tools([get_weather])



In [25]:
response = model_with_tools.invoke("What's the weather like in Boston?") 
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")


content='' additional_kwargs={'reasoning_content': 'The user asks: "What\'s the weather like in Boston?" We need to use the get_weather function. We\'ll call it with location: "Boston".', 'tool_calls': [{'id': 'fc_46f96709-3cd3-4a3a-9a22-db572f5b1516', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 129, 'total_tokens': 183, 'completion_time': 0.05743348, 'completion_tokens_details': {'reasoning_tokens': 31}, 'prompt_time': 0.009801064, 'prompt_tokens_details': None, 'queue_time': 0.346562601, 'total_time': 0.067234544}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a4315eb300', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0cd86-b6cb-7ac3-a24b-5a00edc644a8-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_46f96709-3cd3-4a3a-9a22-db572f5b1516',

## Tool Execution Loops


In [28]:
# step 1: Model generates tool calls
message = [{"role": "user", "content": "What's the weather like in Boston?"}]
ai_msg = model_with_tools.invoke(message)
message.append(ai_msg)

# step 2: Execute tools and Collect results
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    message.append(tool_result)

# step 3: Model generates final response
final_response = model_with_tools.invoke(message)
print(final_response.content)
# The current weather in Boston is 72°F and sunny with a light breeze.


The current weather in Boston is sunny with a temperature of 25 °C. If you need more details (humidity, wind, forecast for the week, etc.), just let me know!


In [29]:
message

[{'role': 'user', 'content': "What's the weather like in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call the get_weather function with location "Boston".', 'tool_calls': [{'id': 'fc_69dd4187-c68f-463a-bcf3-4d351703f162', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 129, 'total_tokens': 166, 'completion_time': 0.040415761, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.006297542, 'prompt_tokens_details': None, 'queue_time': 0.341843803, 'total_time': 0.046713303}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_99996fee8e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0cd8d-3233-7321-89e2-95d8fa2e4d34-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_69dd4187-c68f-463a-bcf3